## lle를 이용한 설비 고장 진단

- 학습용 데이터 분리: 머신러닝을 이용한 설비 고장 진단에서는 고장 데이터가 매우 드물게 발생합니다. 따라서 X_train_normal처럼 정상 데이터로만 모델을 가르치고, 완성된 모델이 고장 데이터를 걸러내도록 설계해야 합니다.
- n_neighbors 하이퍼파라미터: LLE에서 가장 중요한 옵션입니다. 값이 너무 작으면 데이터의 연속성이 끊어지고, 너무 크면 비선형적 굴곡을 무시해 버립니다. 데이터셋의 크기에 맞게 조절해 보세요.
- One-Class SVM의 nu 옵션: nu=0.05는 학습에 쓰인 정상 데이터 중 약 5%는 이상치(노이즈)로 취급하여 무시하겠다는 뜻입니다. 현장의 노이즈 수준에 따라 이 값을 조절하여 오탐률을 제어할 수 있습니다

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

In [2]:
# 1. 가상의 건조기 복합 센서 데이터 생성 (정상 데이터 200개, 고장 데이터 20개)
np.random.seed(42)

# [정상] 온도, 습도, 진동, 전류가 서로 복잡한 비선형 관계를 가지며 순환함
n_normal = 200
t = np.linspace(0, 4 * np.pi, n_normal)
normal_temp = 50 + 20 * np.sin(t) + np.random.normal(0, 1.5, n_normal)
normal_humid = 55 + 35 * np.cos(t) + np.random.normal(0, 1.5, n_normal)
normal_vib = 0.3 + 0.1 * np.sin(t*2) + np.random.normal(0, 0.02, n_normal)
normal_curr = 8 + 3 * np.cos(t/2) + np.random.normal(0, 0.2, n_normal)
X_normal = np.column_stack([normal_temp, normal_humid, normal_vib, normal_curr])

# [고장] 필터 막힘이나 과열 등으로 인해 정상 궤적을 벗어난 이상치 생성
n_fault = 20
fault_temp = np.random.uniform(75, 85, n_fault)
fault_humid = np.random.uniform(60, 70, n_fault)
fault_vib = np.random.uniform(0.4, 0.6, n_fault)
fault_curr = np.random.uniform(10, 13, n_fault)
X_fault = np.column_stack([fault_temp, fault_humid, fault_vib, fault_curr])

# 전체 데이터 병합 (220개 데이터 포인트)
X_all = np.vstack([X_normal, X_fault])
y_true = np.array([1] * n_normal + [-1] * n_fault) # 1: 정상, -1: 고장

print('X_all =', type(X_all), len(X_all), X_all[0])

X_all = <class 'numpy.ndarray'> 220 [50.74507123 90.53668104  0.26811145 11.15139772]


In [ ]:
# 2. 데이터 전처리 (Standard Scaling)
# LLE는 거리 기반이므로 변수들의 스케일(단위)을 반드시 맞춰야 합니다.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

In [ ]:
# 3. LLE를 이용한 차원 축소 (4차원 -> 2차원)
# n_neighbors는 이웃의 수로, 데이터 밀도에 맞게 튜닝이 필요합니다.
lle = LocallyLinearEmbedding(n_neighbors=15, n_components=2, random_state=42)
X_lle = lle.fit_transform(X_scaled)# 3. LLE를 이용한 차원 축소 (4차원 -> 2차원)
# n_neighbors는 이웃의 수로, 데이터 밀도에 맞게 튜닝이 필요합니다.
lle = LocallyLinearEmbedding(n_neighbors=15, n_components=2, random_state=42)
X_lle = lle.fit_transform(X_scaled)

In [ ]:
# 4. 고장 진단 모델 적용 (One-Class SVM)
# 실무에서는 '정상 데이터'만 가지고 학습(fit)을 수행하여 정상 범위를 규정합니다.
X_train_normal = X_lle[:n_normal]
oc_svm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
oc_svm.fit(X_train_normal)

# 전체 데이터(정상+고장)에 대해 고장 여부 예측 (1: 정상, -1: 고장)
y_pred = oc_svm.predict(X_lle)

In [ ]:
# 5. 시각화 (2차원 평면에 정상과 고장 표현)
plt.figure(figsize=(10, 6))

# 실제 레이블을 기준으로 시각화
plt.scatter(X_lle[:n_normal, 0], X_lle[:n_normal, 1], 
            c='blue', label='Normal (True)', alpha=0.6, edgecolor='k')
plt.scatter(X_lle[n_normal:, 0], X_lle[n_normal:, 1], 
            c='red', label='Fault (True)', alpha=0.8, edgecolor='k', marker='X', s=100)

plt.title('Dryer Fault Detection using LLE and One-Class SVM')
plt.xlabel('LLE Component 1')
plt.ylabel('LLE Component 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 6. 간단한 평가 결과 출력
n_false_positive = np.sum((y_true[:n_normal] == 1) & (y_pred[:n_normal] == -1))
n_true_positive_fault = np.sum((y_true[n_normal:] == -1) & (y_pred[n_normal:] == -1))

print(f"[진단 결과]")
print(f"총 정상 데이터 {n_normal}개 중 {n_false_positive}개를 고장으로 오탐(False Alarm)")
print(f"총 고장 데이터 {n_fault}개 중 {n_true_positive_fault}개를 정확히 탐지")